# [6.3] Transcoders and Attribution Graphs - Solutions

This notebook runs the solved transcoder and attribution-graph contracts, then displays the report-backed `gelu-1l` signature result. Keep the claim boundary in view: this is a scoped local preflight, not a published frontier-transcoder replication.

<details>
<summary>Expected output</summary>

The local tests should all print pass messages, and the final table/plots should match the committed `verification_report.json` metrics.

</details>

<details>
<summary>Help - why these controls matter</summary>

Transcoder graphs can look convincing before they are useful. Replacement parity, KL/logit-diff preservation, deterministic graph construction, and top-vs-control damage each remove a different failure mode.

</details>


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter6_sparse_feature_methods"
section = "part3_transcoders_attribution_graphs"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_transcoders_attribution_graphs.tests as tests
import part3_transcoders_attribution_graphs.solutions as solutions

TranscoderOutput = solutions.TranscoderOutput
TranscoderReplacementReport = solutions.TranscoderReplacementReport
AttributionEdge = solutions.AttributionEdge
AttributionGraphReport = solutions.AttributionGraphReport
transcoder_forward = solutions.transcoder_forward
mean_kl_divergence = solutions.mean_kl_divergence
target_logit_diff = solutions.target_logit_diff
transcoder_replacement_report = solutions.transcoder_replacement_report
feature_logit_contributions = solutions.feature_logit_contributions
build_attribution_edges = solutions.build_attribution_edges
graph_reproducible = solutions.graph_reproducible
graph_density = solutions.graph_density
attribution_graph_report = solutions.attribution_graph_report
run_smoke_test = solutions.run_smoke_test


## Replacement Tests

These tests cover the ReLU transcoder forward pass and the behavior-preservation report.


In [ ]:
tests.test_transcoder_forward_matches_reference_and_relu_rules(transcoder_forward)
tests.test_target_logit_diff_and_replacement_report_match_reference(
    target_logit_diff,
    transcoder_replacement_report,
)


## Graph Tests

These tests cover contribution reductions, deterministic edge selection, graph reproducibility, damage controls, and the whole smoke-test contract.


In [ ]:
tests.test_feature_logit_contributions_reduce_all_nonfeature_dimensions(
    feature_logit_contributions,
)
tests.test_build_attribution_edges_keeps_top_input_and_logit_edges(
    build_attribution_edges,
    graph_reproducible,
)
tests.test_graph_reproducible_rejects_structure_and_weight_changes(
    AttributionEdge,
    graph_reproducible,
)
tests.test_attribution_graph_report_preservation_and_damage_controls(
    graph_density,
    attribution_graph_report,
)
tests.test_notebook_contract(run_smoke_test)


## Signature Result

<details>
<summary>Interpreting the signature result</summary>

The oracle path verifies the replacement wiring to numerical precision. The trained tiny transcoder is a lossy replacement that still beats the zero baseline and preserves most top-token behavior on cached activations. The graph result is a controlled hypothesis: top selected features damage the target more than low-effect features, but this is not a full causal circuit proof.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    return [
        ("model", gpu["model_name"]),
        ("HF revision", gpu["hf_revision"][:12]),
        ("prompts / activations", f"{gpu['prompt_count']} / {gpu['activation_count']}"),
        ("d_model / MLP width", f"{gpu['d_model']} / {gpu['mlp_width']}"),
        ("oracle MLP-out max error", f"{gpu['oracle_mlp_out_max_abs_error']:.2e}"),
        ("oracle logits max error", f"{gpu['oracle_logits_max_abs_error']:.2e}"),
        ("tiny transcoder width / steps", f"{gpu['trained_transcoder_width']} / {gpu['trained_transcoder_steps']}"),
        ("held-out MSE / zero MSE", f"{gpu['trained_transcoder_heldout_mse']:.3f} / {gpu['trained_transcoder_heldout_zero_mse']:.3f}"),
        ("held-out MSE ratio", round(gpu["trained_transcoder_heldout_mse_ratio"], 3)),
        ("top-1 agreement", round(gpu["trained_replacement_top1_agreement"], 3)),
        ("feature density", round(gpu["trained_transcoder_feature_density"], 3)),
        ("graph top / low-effect damage", f"{gpu['graph_topk_damage']:.3f} / {gpu['graph_random_damage']:.6f}"),
        ("graph reproducible", gpu["graph_reproducible"]),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["MLP out", "logits"],
    [gpu["oracle_mlp_out_max_abs_error"], gpu["oracle_logits_max_abs_error"]],
    color=["#2563eb", "#0f766e"],
)
axes[0].set_title("Oracle replacement errors")
axes[0].set_ylabel("max abs error")

axes[1].bar(
    ["trained", "zero baseline"],
    [gpu["trained_transcoder_heldout_mse"], gpu["trained_transcoder_heldout_zero_mse"]],
    color=["#16a34a", "#94a3b8"],
)
axes[1].set_title("Held-out reconstruction")
axes[1].set_ylabel("MSE, lower is better")

axes[2].bar(
    ["top graph", "low-effect"],
    [gpu["graph_topk_damage"], gpu["graph_random_damage"]],
    color=["#7c3aed", "#f97316"],
)
axes[2].set_title("Graph damage control")
axes[2].set_ylabel("target logit-diff damage")

fig.tight_layout()
plt.show()


## Limitations

The report proves a scoped local preflight: exact oracle replacement, a tiny trained ReLU transcoder, and one feature-level graph control on `gelu-1l`. It does not prove published-transcoder quality, broad prompt robustness, generated-completion behavior, or a full causal attribution graph.
